# QLoRA: 4-bit Quantized LoRA Fine-tuning

> Implement 4-bit quantization + LoRA for memory-efficient fine-tuning

**QLoRA Paper:** Dettmers et al., 2023 — *QLoRA: Efficient Finetuning of Quantized LLMs*

**Key Components:**
1. **4-bit NormalFloat (NF4)** — Information-theoretically optimal 4-bit quantization
2. **Double Quantization** — Quantize the quantization constants
3. **Paged Optimizers** — Offload optimizer states to CPU when OOM
4. **LoRA** — Low-rank adapters on top of quantized weights

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Optional

## 1. Quantization Fundamentals

In [ ]:
def quantize_symmetric(x: np.ndarray, n_bits: int = 4) -> Tuple[np.ndarray, float]:
    """Symmetric uniform quantization"""
    # Find scale
    abs_max = np.max(np.abs(x))
    scale = abs_max / (2**(n_bits - 1) - 1)
    
    # Quantize
    x_q = np.round(x / scale).astype(np.int8)
    
    # Clamp to range
    x_q = np.clip(x_q, -(2**(n_bits - 1)), 2**(n_bits - 1) - 1)
    
    return x_q, scale

def dequantize_symmetric(x_q: np.ndarray, scale: float) -> np.ndarray:
    """Dequantize symmetric weights"""
    return x_q.astype(np.float32) * scale

def quantize_affine(x: np.ndarray, n_bits: int = 4) -> Tuple[np.ndarray, float, float]:
    """Affine (asymmetric) quantization with zero-point"""
    x_min = np.min(x)
    x_max = np.max(x)
    
    scale = (x_max - x_min) / (2**n_bits - 1)
    zero_point = np.round(-x_min / scale).astype(np.int8)
    
    x_q = np.round(x / scale + zero_point).astype(np.int8)
    x_q = np.clip(x_q, 0, 2**n_bits - 1)
    
    return x_q, scale, zero_point

def dequantize_affine(x_q: np.ndarray, scale: float, zero_point: float) -> np.ndarray:
    """Dequantize affine weights"""
    return (x_q.astype(np.float32) - zero_point) * scale

# Test quantization
W = np.random.randn(256, 256) * 0.5

# Symmetric 4-bit
W_q_sym, scale_sym = quantize_symmetric(W, n_bits=4)
W_dq_sym = dequantize_symmetric(W_q_sym, scale_sym)

# Affine 4-bit
W_q_affine, scale_affine, zp = quantize_affine(W, n_bits=4)
W_dq_affine = dequantize_affine(W_q_affine, scale_affine, zp)

# Error analysis
sym_error = np.mean((W - W_dq_sym)**2)
affine_error = np.mean((W - W_dq_affine)**2)

print(f"Original shape: {W.shape}, dtype: {W.dtype}")
print(f"\nSymmetric 4-bit:")
print(f"  Quantized dtype: {W_q_sym.dtype}, range: [{W_q_sym.min()}, {W_q_sym.max()}]")
print(f"  MSE: {sym_error:.6f}")
print(f"  Memory: {W_q_sym.nbytes / 1024:.2f} KB (vs {W.nbytes / 1024:.2f} KB)")

print(f"\nAffine 4-bit:")
print(f"  Quantized dtype: {W_q_affine.dtype}, range: [{W_q_affine.min()}, {W_q_affine.max()}]")
print(f"  MSE: {affine_error:.6f}")
print(f"  Memory: {W_q_affine.nbytes / 1024:.2f} KB")

## 2. NormalFloat (NF4) Quantization

In [ ]:
def compute_normal_float_quantiles(n_bits: int = 4) -> np.ndarray:
    """
    Compute NF4 quantiles for normally distributed weights.
    NF4 uses quantiles of N(0,1) for optimal information density.
    """
    from scipy import stats
    
    n_quantiles = 2**n_bits
    
    # For symmetric distribution, use quantiles of |N(0,1)|
    # Plus 0 for the center
    if n_bits == 4:
        # NF4 specific quantiles (from QLoRA paper)
        # These are optimized for normal distribution
        quantiles = np.array([
            -1.0, -0.6961928009986877, -0.5250730514526367,
            -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
            -0.09105003625154495, 0.0,
            0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
            0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
            0.7229568362236023, 1.0
        ])
    else:
        # Generic: use quantiles of standard normal
        quantiles = stats.norm.ppf(np.linspace(0.5/n_quantiles, 1 - 0.5/n_quantiles, n_quantiles))
    
    return quantiles

def quantize_nf4(x: np.ndarray, block_size: int = 64) -> Tuple[np.ndarray, np.ndarray]:
    """
    Quantize to NF4 with block-wise scaling.
    """
    nf4_values = compute_normal_float_quantiles(4)
    
    # Reshape to blocks
    original_shape = x.shape
    x_flat = x.flatten()
    n_blocks = int(np.ceil(len(x_flat) / block_size))
    
    # Pad if needed
    pad_len = n_blocks * block_size - len(x_flat)
    if pad_len > 0:
        x_flat = np.concatenate([x_flat, np.zeros(pad_len)])
    
    x_blocks = x_flat.reshape(n_blocks, block_size)
    
    # Quantize each block
    x_q = np.zeros((n_blocks, block_size), dtype=np.uint8)
    absmax = np.zeros(n_blocks, dtype=np.float32)
    
    for i in range(n_blocks):
        block = x_blocks[i]
        amax = np.max(np.abs(block))
        absmax[i] = amax
        
        if amax > 0:
            normalized = block / amax
            # Find nearest NF4 value
            for j in range(block_size):
                idx = np.argmin(np.abs(nf4_values - normalized[j]))
                x_q[i, j] = idx
    
    return x_q, absmax

def dequantize_nf4(x_q: np.ndarray, absmax: np.ndarray, block_size: int = 64,
                   original_shape: tuple = None) -> np.ndarray:
    """Dequantize NF4 weights"""
    nf4_values = compute_normal_float_quantiles(4)
    
    n_blocks = x_q.shape[0]
    x_dq = np.zeros((n_blocks, block_size), dtype=np.float32)
    
    for i in range(n_blocks):
        x_dq[i] = nf4_values[x_q[i]] * absmax[i]
    
    x_dq = x_dq.flatten()
    
    if original_shape:
        total = np.prod(original_shape)
        x_dq = x_dq[:total].reshape(original_shape)
    
    return x_dq

# Test NF4
W_nf4_q, absmax = quantize_nf4(W, block_size=64)
W_nf4_dq = dequantize_nf4(W_nf4_q, absmax, block_size=64, original_shape=W.shape)

nf4_error = np.mean((W - W_nf4_dq)**2)

print(f"NF4 Quantization:")
print(f"  Block size: 64")
print(f"  MSE: {nf4_error:.6f}")
print(f"  Quantized size: {W_nf4_q.nbytes / 1024:.2f} KB")
print(f"  Absmax size: {absmax.nbytes / 1024:.2f} KB")
print(f"  Total: {(W_nf4_q.nbytes + absmax.nbytes) / 1024:.2f} KB")
print(f"  Compression: {W.nbytes / (W_nf4_q.nbytes + absmax.nbytes):.1f}x")

## 3. Double Quantization

In [ ]:
def double_quantize(absmax: np.ndarray, n_bits: int = 8) -> Tuple[np.ndarray, float]:
    """
    Quantize the quantization constants (absmax values).
    This further reduces memory overhead.
    """
    absmax_q, scale = quantize_symmetric(absmax, n_bits=n_bits)
    return absmax_q, scale

def double_dequantize(absmax_q: np.ndarray, scale: float) -> np.ndarray:
    """Dequantize the quantization constants"""
    return dequantize_symmetric(absmax_q, scale)

# Test double quantization
absmax_q, dq_scale = double_quantize(absmax, n_bits=8)
absmax_dq = double_dequantize(absmax_q, dq_scale)

dq_error = np.mean((absmax - absmax_dq)**2)

print(f"Double Quantization:")
print(f"  Original absmax: {absmax.nbytes} bytes")
print(f"  Quantized absmax: {absmax_q.nbytes} bytes")
print(f"  Scale: 4 bytes")
print(f"  MSE: {dq_error:.8f}")
print(f"  Additional compression: {absmax.nbytes / (absmax_q.nbytes + 4):.1f}x")

## 4. QLoRA Layer: 4-bit Weights + LoRA Adapters

In [ ]:
class QLoRALayer:
    """
    QLoRA: 4-bit quantized base weights + LoRA adapters.
    Only LoRA parameters (B, A) are in full precision and trainable.
    """
    
    def __init__(self, in_features: int, out_features: int, rank: int = 64,
                 alpha: float = 16, block_size: int = 64):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.block_size = block_size
        
        # Base weights (4-bit quantized, frozen)
        W_base = np.random.randn(out_features, in_features) * 0.02
        self.W_q, self.absmax = quantize_nf4(W_base, block_size)
        self.absmax_q, self.absmax_scale = double_quantize(self.absmax)
        
        # LoRA adapters (full precision, trainable)
        self.B = np.zeros((out_features, rank))
        self.A = np.random.randn(rank, in_features) * 0.01
    
    def get_base_weight(self) -> np.ndarray:
        """Dequantize base weight for forward pass"""
        absmax_dq = double_dequantize(self.absmax_q, self.absmax_scale)
        W_dq = dequantize_nf4(self.W_q, absmax_dq, self.block_size,
                             (self.out_features, self.in_features))
        return W_dq
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """Forward: h = x @ W_4bit^T + (x @ A^T @ B^T) * scaling"""
        # Dequantize on-the-fly (in practice, done in GPU kernels)
        W_base = self.get_base_weight()
        
        # Base output (from quantized weights)
        base_out = x @ W_base.T
        
        # LoRA output
        lora_out = (x @ self.A.T @ self.B.T) * self.scaling
        
        return base_out + lora_out
    
    def memory_usage(self) -> dict:
        """Calculate memory usage breakdown"""
        base_quantized = self.W_q.nbytes + self.absmax_q.nbytes + 4  # +scale
        lora_trainable = self.B.nbytes + self.A.nbytes
        
        # If stored in full precision (for comparison)
        base_fp16 = self.out_features * self.in_features * 2
        base_fp32 = self.out_features * self.in_features * 4
        
        return {
            'base_quantized': base_quantized,
            'lora_trainable': lora_trainable,
            'total_qlora': base_quantized + lora_trainable,
            'full_fp16': base_fp16,
            'full_fp32': base_fp32,
            'savings_fp16': base_fp16 / (base_quantized + lora_trainable),
            'savings_fp32': base_fp32 / (base_quantized + lora_trainable)
        }

# Test QLoRA
qlora = QLoRALayer(in_features=4096, out_features=4096, rank=64, block_size=64)
mem = qlora.memory_usage()

print(f"QLoRA Layer (4096 x 4096):")
print(f"  Base (quantized): {mem['base_quantized'] / 1024:.2f} KB")
print(f"  LoRA adapters: {mem['lora_trainable'] / 1024:.2f} KB")
print(f"  Total QLoRA: {mem['total_qlora'] / 1024:.2f} KB")
print(f"  Full FP16: {mem['full_fp16'] / 1024:.2f} KB")
print(f"  Full FP32: {mem['full_fp32'] / 1024:.2f} KB")
print(f"  Savings vs FP16: {mem['savings_fp16']:.1f}x")
print(f"  Savings vs FP32: {mem['savings_fp32']:.1f}x")

# Forward test
x = np.random.randn(2, 4096)
out = qlora.forward(x)
print(f"\nInput: {x.shape}, Output: {out.shape}")

## 5. Memory Comparison: Full vs LoRA vs QLoRA

In [ ]:
def estimate_model_memory(
    model_size: str = "7B",
    batch_size: int = 1,
    seq_length: int = 512,
    method: str = "full"
) -> dict:
    """Estimate memory usage for different fine-tuning methods"""
    
    # Model parameters (approximate)
    params_map = {"1B": 1e9, "7B": 7e9, "13B": 13e9, "70B": 70e9}
    n_params = params_map.get(model_size, 7e9)
    
    # Bytes per parameter
    if method == "full":
        param_bytes = 4  # FP32 training
        grad_bytes = 4
        optimizer_bytes = 8  # Adam: 2x params
        activation_bytes = 4 * batch_size * seq_length * 4096 / n_params * 4  # Rough
        
        model_mem = n_params * param_bytes
        grad_mem = n_params * grad_bytes
        opt_mem = n_params * optimizer_bytes
        total = model_mem + grad_mem + opt_mem
        
    elif method == "lora":
        # Only 0.1% trainable
        trainable_ratio = 0.001
        model_mem = n_params * 2  # FP16
        grad_mem = n_params * trainable_ratio * 4
        opt_mem = n_params * trainable_ratio * 8
        total = model_mem + grad_mem + opt_mem
        
    elif method == "qlora":
        # 4-bit base + FP16 LoRA
        trainable_ratio = 0.001
        model_mem = n_params * 0.5  # ~4-bit avg
        grad_mem = n_params * trainable_ratio * 4
        opt_mem = n_params * trainable_ratio * 8
        # Paged optimizer overhead
        page_overhead = n_params * trainable_ratio * 8 * 0.1
        total = model_mem + grad_mem + opt_mem + page_overhead
    
    return {
        'model': model_mem / 1e9,
        'gradients': grad_mem / 1e9,
        'optimizer': opt_mem / 1e9,
        'total': total / 1e9,
    }

# Compare for different model sizes
models = ["1B", "7B", "13B", "70B"]
methods = ["full", "lora", "qlora"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, model_size in enumerate(models):
    ax = axes[idx // 2, idx % 2]
    
    mem_data = {m: estimate_model_memory(model_size, method=m) for m in methods}
    
    x = np.arange(len(methods))
    width = 0.25
    
    model_vals = [mem_data[m]['model'] for m in methods]
    grad_vals = [mem_data[m]['gradients'] for m in methods]
    opt_vals = [mem_data[m]['optimizer'] for m in methods]
    
    ax.bar(x - width, model_vals, width, label='Model', color='#FF6B6B')
    ax.bar(x, grad_vals, width, label='Gradients', color='#4ECDC4')
    ax.bar(x + width, opt_vals, width, label='Optimizer', color='#45B7D1')
    
    ax.set_ylabel('Memory (GB)')
    ax.set_title(f'{model_size} Model Memory Breakdown')
    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("\n📊 Memory Requirements (GB):")
print("=" * 60)
print(f"{'Model':<8} {'Full FT':<12} {'LoRA':<12} {'QLoRA':<12} {'GPU Needed'}")
print("-" * 60)

gpu_reqs = {"full": "A100 80GB", "lora": "A100 40GB", "qlora": "RTX 3090"}

for model_size in models:
    mem_data = {m: estimate_model_memory(model_size, method=m) for m in methods}
    print(f"{model_size:<8} {mem_data['full']['total']:<12.1f} {mem_data['lora']['total']:<12.1f} {mem_data['qlora']['total']:<12.1f}")

## 6. Quantization Error Analysis

In [ ]:
# Compare quantization methods
np.random.seed(42)
test_weights = np.random.randn(512, 512) * 0.5

methods_comparison = {}

# FP32 baseline
methods_comparison['FP32'] = {
    'dequantized': test_weights.copy(),
    'bytes': test_weights.nbytes
}

# FP16
W_fp16 = test_weights.astype(np.float16).astype(np.float32)
methods_comparison['FP16'] = {
    'dequantized': W_fp16,
    'bytes': test_weights.size * 2
}

# INT8 symmetric
W_q8, s8 = quantize_symmetric(test_weights, 8)
W_dq8 = dequantize_symmetric(W_q8, s8)
methods_comparison['INT8'] = {
    'dequantized': W_dq8,
    'bytes': test_weights.size * 1 + 4
}

# INT4 symmetric
W_q4, s4 = quantize_symmetric(test_weights, 4)
W_dq4 = dequantize_symmetric(W_q4, s4)
methods_comparison['INT4'] = {
    'dequantized': W_dq4,
    'bytes': test_weights.size * 0.5 + 4
}

# NF4
W_nf4_q, nf4_absmax = quantize_nf4(test_weights, block_size=64)
W_nf4_dq = dequantize_nf4(W_nf4_q, nf4_absmax, block_size=64, original_shape=test_weights.shape)
methods_comparison['NF4'] = {
    'dequantized': W_nf4_dq,
    'bytes': W_nf4_q.nbytes + nf4_absmax.nbytes
}

# Calculate errors
results = []
for name, data in methods_comparison.items():
    mse = np.mean((test_weights - data['dequantized'])**2)
    max_err = np.max(np.abs(test_weights - data['dequantized']))
    compression = test_weights.nbytes / data['bytes']
    results.append({
        'method': name,
        'mse': mse,
        'max_error': max_err,
        'bytes': data['bytes'],
        'compression': compression
    })

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

names = [r['method'] for r in results]

# MSE
mses = [r['mse'] for r in results]
axes[0].bar(names, mses, color=['#2d333b', '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'], edgecolor='black')
axes[0].set_ylabel('MSE')
axes[0].set_title('Quantization Error (MSE)')
axes[0].set_yscale('log')
axes[0].grid(axis='y', alpha=0.3)

# Max error
max_errs = [r['max_error'] for r in results]
axes[1].bar(names, max_errs, color=['#2d333b', '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'], edgecolor='black')
axes[1].set_ylabel('Max Absolute Error')
axes[1].set_title('Maximum Quantization Error')
axes[1].grid(axis='y', alpha=0.3)

# Compression ratio
comps = [r['compression'] for r in results]
axes[2].bar(names, comps, color=['#2d333b', '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'], edgecolor='black')
axes[2].set_ylabel('Compression Ratio')
axes[2].set_title('Memory Compression')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Quantization Comparison:")
print(f"{'Method':<10} {'MSE':<15} {'Max Error':<12} {'Bytes':<10} {'Compression'}")
print("-" * 65)
for r in results:
    print(f"{r['method']:<10} {r['mse']:<15.6f} {r['max_error']:<12.4f} {r['bytes']:<10} {r['compression']:.1f}x")

## 🎯 Exercises

1. **Block Size Impact**: Test block sizes 32, 64, 128, 256. Plot error vs compression.
2. **Outlier Handling**: Implement outlier-aware quantization (keep top 0.1% in FP16).
3. **Mixed Precision**: Quantize only certain layers (attention vs FFN).
4. **Activation Quantization**: Quantize activations in addition to weights.
5. **GPTQ Integration**: Combine QLoRA with GPTQ for even better quantization.
6. **Real Model**: Use `bitsandbytes` library to quantize a real model.